# Eksperimen dan Modeling Logistic Regression — Deteksi Situs Web Phishing
**PIC:** Rapip (Kelompok 4)  
**Dataset:** `dataset/output/` — 4.679 train, 1.170 test, 30 fitur  
**Target:** `0` = Legitimate / Aman, `1` = Phishing

Tahapan mengikuti notebook KNN: memuat data dan standardisasi, mencari parameter,
melatih dan mengevaluasi model terbaik, serta menyimpan model dan metrik.
Pada Logistic Regression, **C** mengatur kekuatan regularisasi; **koefisien β**
dipelajari otomatis saat pelatihan. C dipilih melalui validasi silang pada train.

VS Code: pilih kernel Python lalu **Run All**. Colab: unggah notebook dan sediakan
folder repositori, lalu isi `REPO_ROOT` pada sel dataset jika diperlukan.


## Dependensi
Jalankan sel berikut jika paket belum tersedia. Gunakan scikit-learn 1.8.0 agar sesuai dengan scaler bersama; restart kernel setelah mengganti versi.

In [ ]:
# Jalankan sekali bila diperlukan dengan menghapus tanda #:
# %pip install scikit-learn==1.8.0 pandas numpy joblib matplotlib

In [ ]:
import json
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, classification_report,
)


## 1 Memuat Dataset dan Standardisasi Fitur
Gunakan split bersama tanpa membagi ulang data. Scaler tersimpan hanya di-fit pada train. Target output sudah biner dan tidak dipetakan ulang.

In [ ]:
REPO_ROOT = Path.cwd()  # Colab: ganti dengan lokasi repositori yang tersedia
if not (REPO_ROOT / "dataset/output").exists():
    REPO_ROOT = REPO_ROOT.parent  # Jika dijalankan dari folder notebooks
DATA_DIR = REPO_ROOT / "dataset/output"
MODEL_DIR = REPO_ROOT / "models"
EVAL_DIR = REPO_ROOT / "evaluation"
MODEL_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(exist_ok=True)

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze("columns")
assert list(X_train.columns) == list(X_test.columns)
assert set(y_train.unique()) == set(y_test.unique()) == {0, 1}

scaler = joblib.load(DATA_DIR / "scaler.pkl")
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"X_train: {X_train_scaled.shape} | X_test: {X_test_scaled.shape}")
print("Proporsi y_test (0=Legitimate, 1=Phishing):", y_test.value_counts().to_dict())


## 2 Eksperimen Pencarian Nilai C Terbaik
Menguji C = 0,001; 0,01; 0,1; 1; 10; 100; 1.000 dengan 5-fold CV. C lebih besar berarti regularisasi lebih lemah. Pemilihan memakai rata-rata F1 phishing tertinggi. Pipeline melakukan fitting scaler di dalam setiap fold agar data validasi tidak ikut menentukan standardisasi.

In [ ]:
lr_params = dict(solver="lbfgs", l1_ratio=0, max_iter=5000, tol=1e-8, random_state=42)
search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(**lr_params))]),
    {"lr__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000]},
    scoring={"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"},
    refit="f1", cv=StratifiedKFold(5, shuffle=True, random_state=42), n_jobs=1,
)
search.fit(X_train, y_train)
cv_results = pd.DataFrame(search.cv_results_)
results_df = pd.DataFrame({
    "C": cv_results["param_lr__C"].astype(float),
    "Accuracy": cv_results["mean_test_accuracy"],
    "Precision_Phishing": cv_results["mean_test_precision"],
    "Recall_Phishing": cv_results["mean_test_recall"],
    "F1_Phishing": cv_results["mean_test_f1"],
    "ROC_AUC": cv_results["mean_test_roc_auc"],
})
display(results_df.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for metric in ["Accuracy", "F1_Phishing", "Recall_Phishing", "ROC_AUC"]:
    ax.plot(results_df["C"], results_df[metric], marker="o", label=metric)
ax.set(xscale="log", xlabel="C", ylabel="Rata-rata skor validasi", title="Perbandingan Logistic Regression pada CV Train")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 3 Pelatihan dan Evaluasi Model Terbaik
Latih model final dengan C terpilih dan scaler bersama. Ambang prediksi adalah 0,5. Koefisien positif menaikkan log odds phishing saat nilai fitur meningkat; koefisien negatif menurunkannya, dengan fitur lain tetap.

In [ ]:
best_C = float(search.best_params_["lr__C"])
print(f"Nilai C terbaik: {best_C:g}")
best_model = LogisticRegression(C=best_C, **lr_params)
t0 = time.perf_counter()
best_model.fit(X_train_scaled, y_train)
train_time = time.perf_counter() - t0
y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, list(best_model.classes_).index(1)]

coefficients = pd.DataFrame({"Fitur": X_train.columns, "Koefisien standar": best_model.coef_[0]})
print("Intercept:", best_model.intercept_[0])
display(coefficients.round(6))
print(classification_report(y_test, y_pred, target_names=["Legitimate (0)", "Phishing (1)"], digits=4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=["Legitimate (0)", "Phishing (1)"], cmap="Blues", ax=axes[0])
axes[0].set_title(f"Confusion Matrix — Logistic Regression C={best_C:g}")
RocCurveDisplay.from_predictions(y_test, y_proba, name="Logistic Regression", ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Guess")
axes[1].set_title("ROC Curve")
axes[1].legend()
plt.tight_layout()
plt.show()


## 4 Simpan Model dan Metrik Evaluasi
Model menerima fitur yang telah ditransformasi menggunakan scaler bersama. Metrik menggunakan kelas phishing (1) sebagai kelas positif dan format yang sama dengan notebook KNN.

In [ ]:
model_path = MODEL_DIR / "logistic_regression_model.pkl"
joblib.dump(best_model, model_path)
metrics_summary = {
    "Algoritma": "Logistic Regression", "PIC": "Rapip",
    "best_params": {"C": best_C, "solver": "lbfgs", "l1_ratio": 0},
    "Accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "Precision (Phishing)": round(float(precision_score(y_test, y_pred, pos_label=1)), 4),
    "Recall (Phishing)": round(float(recall_score(y_test, y_pred, pos_label=1)), 4),
    "F1-Score (Phishing)": round(float(f1_score(y_test, y_pred, pos_label=1)), 4),
    "ROC-AUC": round(float(roc_auc_score(y_test, y_proba)), 4),
    "Train Time (s)": round(float(train_time), 4),
}
metrics_path = EVAL_DIR / "logistic_regression_metrics.json"
metrics_path.write_text(json.dumps(metrics_summary, indent=2), encoding="utf-8")
print("Model tersimpan:", model_path.resolve())
print("Metrik tersimpan:", metrics_path.resolve())
print(json.dumps(metrics_summary, indent=2))
